In [ ]:
#========================================================
#show logarighim image and histogram
#========================================================

import sys
import os
import matplotlib.pyplot as plt
import numpy as np

# 경로 설정 (사용자 환경에 맞게 유지)
root_path = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.generals.utils import load_hdr
from src.gradient_tone_mapping.tonemap import tonemap, _step1_log_mapping

# 데이터 로드
input_path = "../../data/image_sample/3_Boundary_Halo/01_3072 x 2048_pos(1)_NG.hdr"
hdr = load_hdr(input_path)

# Step 1: Log Mapping 적용
log_map = _step1_log_mapping(hdr)

# 시각화를 위한 정규화 (0-255)
log_display = (log_map * 255)
log_display = np.clip(log_display, 0, 255).astype(np.uint8)

# 시각화: 이미지와 히스토그램을 가로로 배치
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. 로그 변환 이미지 출력
axes[0].imshow(log_display, cmap='gray')
axes[0].set_title('Logarithmized Image (Step 1)')
axes[0].axis('off')

# 2. 히스토그램 출력
# ravel()을 사용하여 2차원 배열을 1차원으로 펼쳐서 계산합니다.
axes[1].hist(log_map.ravel(), bins=256, color='gray', alpha=0.7)
axes[1].set_title('Histogram of Log Map (Raw)')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
#========================================================
#show gradient map
#========================================================
import sys
import os
import matplotlib.pyplot as plt
import numpy as np

# path setting
root_path = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.generals.utils import load_hdr
from src.gradient_tone_mapping.parameters import Parameters 
from src.gradient_tone_mapping.tonemap import tonemap, _step1_log_mapping, generic_filter, _local_fuzzy_entropy, _attenuation, _compressed_gradient
from src.gradient_tone_mapping.gradient import compute_gradient_central_difference

params = Parameters()
gamma = 0.3
# data load
input_path = "../../data/image_sample/3_Boundary_Halo/01_3072 x 2048_pos(1)_NG.hdr"
hdr = load_hdr(input_path)
log_map = _step1_log_mapping(hdr)

def get_logarithm_gradient(log_map):
    gx,gy =compute_gradient_central_difference(log_map)
    return np.sqrt(gx**2 + gy**2)
    

def get_ideal_gradient_map(log_map, gamma):
    print(f"gamma = {gamma}")

    E            = generic_filter(log_map, _local_fuzzy_entropy, size=params.neighbor_size) # Eq. (6), (7)
    K            = _attenuation(E, gamma) # Eq. (8)

    # ============
    # for exp: check current issues (while image, many noise) ..
    print(f"\nE min = {E.min():.6f}, E max = {E.max():.6f}, E mean = {E.mean():.6f}")
    print(f"\nK min = {K.min():.2f}, K max = {K.max():.2f}, K mean = {K.mean():.2f}")
    # ============

    Gx, Gy       = _compressed_gradient(log_map, K) # Eq. (2)
    G_map = np.sqrt(Gx**2 + Gy**2)
    return G_map

log_G_map = get_logarithm_gradient(log_map)
ideal_G_map = get_ideal_gradient_map(log_map, gamma)

fig, axes = plt.subplots(1, 2, figsize= (12, 5))
fig.suptitle("original logarithm domain image gradeint map vs ehanced ideal gradient map", fontsize=16)
axes[0].imshow(log_G_map, cmap='gray')
axes[0].set_title('Original logarithm domain')
axes[0].axis('off')

axes[1].imshow(ideal_G_map, cmap='gray')
axes[1].set_title('ideal gradient')
axes[1].axis('off')
plt.tight_layout()
plt.show()

